## Section 1: Imports

In [2]:
import pandas as pd
import numpy as np
import joblib
import json

from pathlib import Path

pd.set_option("display.max_columns", None)

## Section 2: Load Processed Data

In [3]:
model_df = pd.read_parquet(
    "../data/processed/model_df.parquet"
)

product_features = pd.read_parquet(
    "../artifacts/product_features.parquet"
)

popularity_df = pd.read_parquet(
    "../artifacts/popularity_recommender.parquet"
)

## Section 3: Train/Test Split

In [4]:
split_date = "2018-06-30"

train_df = model_df[
    model_df["order_purchase_timestamp"] <= split_date
].copy()

test_df = model_df[
    model_df["order_purchase_timestamp"] > split_date
].copy()

print(train_df.shape)
print(test_df.shape)

(61419, 5)
(7391, 5)


## Section 4: Create Customer Features

### Total Spend

In [5]:
customer_spend = (
    train_df
    .groupby("customer_unique_id")["price"]
    .sum()
    .reset_index()
    .rename(
        columns={
            "price": "customer_total_spend"
        }
    )
)

In [6]:
customer_spend.head()

,customer_unique_id,customer_total_spend
0,0000366f3b9a7992bf8c76cfdf3221e2,129.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,18.90
2,0000f46a3911fa3c0805444483337064,69.00
3,00050ab1314c0e55a6ca13cf7181fecf,27.99
4,00053a61a98854899e70ed204dd4bafe,191.00


### Purchase Frequency

In [7]:
customer_frequency = (
    train_df
    .groupby("customer_unique_id")
    .size()
    .reset_index(name="customer_purchase_count")
)

### Favorite Category

In [8]:
customer_category = (
    train_df
    .groupby(
        [
            "customer_unique_id",
            "product_category_name"
        ]
    )
    .size()
    .reset_index(name="cnt")
)

customer_favorite_category = (
    customer_category
    .sort_values(
        "cnt",
        ascending=False
    )
    .drop_duplicates(
        subset=["customer_unique_id"]
    )
)

customer_favorite_category = (
    customer_favorite_category[
        [
            "customer_unique_id",
            "product_category_name"
        ]
    ]
    .rename(
        columns={
            "product_category_name":
            "favorite_category"
        }
    )
)

## Section 5: Merge Customer Features

In [9]:
customer_features = (
    customer_spend
    .merge(
        customer_frequency,
        on="customer_unique_id"
    )
    .merge(
        customer_favorite_category,
        on="customer_unique_id"
    )
)

customer_features.head()

,customer_unique_id,customer_total_spend,customer_purchase_count,favorite_category
0,0000366f3b9a7992bf8c76cfdf3221e2,129.90,1,cama_mesa_banho
1,0000b849f77a49e4a4ce2b2a4ca5be3f,18.90,1,beleza_saude
2,0000f46a3911fa3c0805444483337064,69.00,1,papelaria
3,00050ab1314c0e55a6ca13cf7181fecf,27.99,1,telefonia
4,00053a61a98854899e70ed204dd4bafe,191.00,1,esporte_lazer


## Section 6: Create Product Features

### Popularity scores

In [10]:
product_features_rank = (
    product_features
    .merge(
        popularity_df[
            [
                "product_id",
                "popularity_score"
            ]
        ],
        on="product_id",
        how="left"
    )
)

### Fill missing popularity

In [11]:
product_features_rank[
    "popularity_score"
] = (
    product_features_rank[
        "popularity_score"
    ].fillna(0)
)

In [12]:
product_features_rank.head()

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm,popularity_score
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,225.0,16.0,10.0,14.0,0.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,1000.0,30.0,18.0,20.0,0.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,154.0,18.0,9.0,15.0,0.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,371.0,26.0,4.0,26.0,0.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,625.0,20.0,17.0,13.0,0.0


## Section 7: Positive Samples

In [13]:
positive_samples = (
    train_df[
        [
            "customer_unique_id",
            "product_id",
            "product_category_name"
        ]
    ]
    .copy()
)

positive_samples = positive_samples.dropna(
    subset=["product_category_name"]
)

positive_samples["label"] = 1

print(positive_samples.shape)

print(
    positive_samples["product_category_name"]
    .isna()
    .sum()
)

(60648, 4)
0


In [14]:
positive_samples.shape

(60648, 4)

In [15]:
print(customer_features.shape)

print(product_features_rank.shape)

print(positive_samples.shape)

(51613, 4)
(32951, 7)
(60648, 4)


## Section 8: Create Hard Negatives

### Build Customer Purchase History

In [16]:
customer_purchases = (
    train_df
    .groupby("customer_unique_id")["product_id"]
    .apply(set)
    .to_dict()
)

### Build Category → Products Mapping

In [17]:
category_products = (
    train_df
    .groupby("product_category_name")["product_id"]
    .unique()
    .to_dict()
)

## Section 9: Generate Hard Negatives

In [18]:
np.random.seed(42)

negative_samples = []

for row in positive_samples.itertuples(index=False):

    customer_id = row.customer_unique_id
    category = row.product_category_name

    purchased_products = customer_purchases[customer_id]

    candidate_products = (
        set(category_products[category])
        - purchased_products
    )

    if len(candidate_products) == 0:
        continue

    n_negatives = min(3, len(candidate_products))

    sampled_negatives = np.random.choice(
        list(candidate_products),
        size=n_negatives,
        replace=False
    )

    for product in sampled_negatives:

        negative_samples.append(
            {
                "customer_unique_id": customer_id,
                "product_id": product,
                "product_category_name": category,
                "label": 0
            }
        )

### Convert To DataFrame

In [19]:
negative_samples_df = pd.DataFrame(
    negative_samples
)

print(negative_samples_df.shape)

(181466, 4)


## Section 10: Build Ranking Dataset

In [20]:
ranking_df = pd.concat(
    [
        positive_samples,
        negative_samples_df
    ],
    ignore_index=True
)

ranking_df = ranking_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [21]:
print(ranking_df.shape)

print(
    ranking_df["label"]
    .value_counts()
)

(242114, 4)
label
0    181466
1     60648
Name: count, dtype: int64


In [22]:
print(positive_samples.shape)
print(negative_samples_df.shape)
print(ranking_df["label"].value_counts())

(60648, 4)
(181466, 4)
label
0    181466
1     60648
Name: count, dtype: int64


## Section 11: Merge Customer Features

In [23]:
ranking_df = ranking_df.merge(
    customer_features,
    on="customer_unique_id",
    how="left"
)

print(ranking_df.shape)

(242114, 7)


## Section 12: Merge Product Features

In [24]:
ranking_df = ranking_df.merge(
    product_features_rank,
    on="product_id",
    how="left"
)

print(ranking_df.shape)

(242114, 13)


In [25]:
print(ranking_df.columns.tolist())

['customer_unique_id', 'product_id', 'product_category_name_x', 'label', 'customer_total_spend', 'customer_purchase_count', 'favorite_category', 'product_category_name_y', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'popularity_score']


In [26]:
[col for col in ranking_df.columns if "category" in col]

['product_category_name_x', 'favorite_category', 'product_category_name_y']

In [27]:
ranking_df = ranking_df.rename(
    columns={
        "product_category_name_y": "product_category_name"
    }
)

In [28]:
if "product_category_name_x" in ranking_df.columns:
    ranking_df = ranking_df.drop(
        columns=["product_category_name_x"]
    )

In [29]:
print(
    [col for col in ranking_df.columns if "category" in col]
)

['favorite_category', 'product_category_name']


## Section 13: Create Interaction Feature

In [30]:
ranking_df["is_favorite_category_match"] = (
    ranking_df["favorite_category"]
    ==
    ranking_df["product_category_name"]
).astype(int)

In [31]:
ranking_df[
    [
        "favorite_category",
        "product_category_name",
        "is_favorite_category_match"
    ]
].head(20)

,favorite_category,product_category_name,is_favorite_category_match
0,esporte_lazer,esporte_lazer,1
1,beleza_saude,beleza_saude,1
2,relogios_presentes,relogios_presentes,1
3,consoles_games,consoles_games,1
4,moveis_decoracao,moveis_decoracao,1
5,eletronicos,eletronicos,1
6,eletronicos,eletronicos,1
7,telefonia,telefonia,1
8,informatica_acessorios,informatica_acessorios,1
9,ferramentas_jardim,ferramentas_jardim,1


## Section 14: Encode Categorical Variables

In [32]:
from sklearn.preprocessing import LabelEncoder
customer_encoder = LabelEncoder()

ranking_df["customer_idx"] = (
    customer_encoder.fit_transform(
        ranking_df["customer_unique_id"]
    )
)

In [33]:
product_encoder = LabelEncoder()

ranking_df["product_idx"] = (
    product_encoder.fit_transform(
        ranking_df["product_id"]
    )
)

In [34]:
category_encoder = LabelEncoder()

ranking_df["category_idx"] = (
    category_encoder.fit_transform(
        ranking_df["product_category_name"]
    )
)

In [35]:
ranking_df["favorite_category_idx"] = (
    category_encoder.transform(
        ranking_df["favorite_category"]
    )
)

## Section 15: Final Feature List

In [36]:
FEATURE_COLUMNS = [
    "customer_total_spend",
    "customer_purchase_count",

    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",

    "popularity_score",

    "is_favorite_category_match",

    "customer_idx",
    "product_idx",
    "category_idx",
    "favorite_category_idx"
]

In [37]:
ranking_df[FEATURE_COLUMNS].head()

,customer_total_spend,customer_purchase_count,product_weight_g,product_length_cm,product_height_cm,product_width_cm,popularity_score,is_favorite_category_match,customer_idx,product_idx,category_idx,favorite_category_idx
0,29.9,1,800.0,50.0,15.0,25.0,0.040230,1,3107,4571,31,31
1,150.0,1,400.0,30.0,10.0,17.0,0.009579,1,47266,3185,10,10
2,55.0,1,333.0,18.0,12.0,17.0,0.005747,1,14778,4105,61,61
3,49.0,2,150.0,16.0,12.0,12.0,0.268199,1,48671,199,19,19
4,53.4,6,1050.0,25.0,25.0,25.0,0.022989,1,28829,4154,51,51


## Section 16: Missing Value Check

In [38]:
print(
    ranking_df[FEATURE_COLUMNS]
    .isna()
    .sum()
)

customer_total_spend          0
customer_purchase_count       0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
popularity_score              0
is_favorite_category_match    0
customer_idx                  0
product_idx                   0
category_idx                  0
favorite_category_idx         0
dtype: int64


In [39]:
print(ranking_df.shape)

print(
    ranking_df[FEATURE_COLUMNS]
    .isna()
    .sum()
)

(242114, 17)
customer_total_spend          0
customer_purchase_count       0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
popularity_score              0
is_favorite_category_match    0
customer_idx                  0
product_idx                   0
category_idx                  0
favorite_category_idx         0
dtype: int64


## Section 18: Imports

In [40]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

## Section 19: Train/Validation Split

In [41]:
train_rank_df, valid_rank_df = train_test_split(
    ranking_df,
    test_size=0.2,
    random_state=42,
    stratify=ranking_df["label"]
)

In [42]:
print(train_rank_df.shape)
print(valid_rank_df.shape)

(193691, 17)
(48423, 17)


## Section 20: Build Feature Matrices

In [43]:
X_train = train_rank_df[FEATURE_COLUMNS]
y_train = train_rank_df["label"]

X_valid = valid_rank_df[FEATURE_COLUMNS]
y_valid = valid_rank_df["label"]

In [44]:
print(X_train.shape)
print(X_valid.shape)

(193691, 12)
(48423, 12)


## Section 21: Create Group Information

In [45]:
train_groups = (
    train_rank_df
    .groupby("customer_idx")
    .size()
    .values
)

valid_groups = (
    valid_rank_df
    .groupby("customer_idx")
    .size()
    .values
)

In [46]:
print(len(train_groups))
print(len(valid_groups))

print(train_groups[10:20])

51525
32068
[3 3 3 8 2 3 2 7 3 4]


In [47]:
print(X_train.shape)
print(X_valid.shape)

print(len(train_groups))
print(len(valid_groups))

print(train_groups[10:20])

(193691, 12)
(48423, 12)
51525
32068
[3 3 3 8 2 3 2 7 3 4]


#### We did train/test split by row, We must split by customer, not by rows.

In [48]:
unique_customers = ranking_df[
    "customer_unique_id"
].unique()

len(unique_customers)

51613

## Split Customers

In [49]:
from sklearn.model_selection import train_test_split

unique_customers = (
    ranking_df["customer_unique_id"]
    .unique()
    .to_numpy()
)

train_customers, valid_customers = train_test_split(
    unique_customers,
    test_size=0.2,
    random_state=42
)

In [50]:
train_rank_df = ranking_df[
    ranking_df["customer_unique_id"]
    .isin(train_customers)
].copy()

valid_rank_df = ranking_df[
    ranking_df["customer_unique_id"]
    .isin(valid_customers)
].copy()

## Build Datasets

In [51]:
train_rank_df = train_rank_df.sort_values(
    "customer_idx"
).reset_index(drop=True)

valid_rank_df = valid_rank_df.sort_values(
    "customer_idx"
).reset_index(drop=True)

## Recreate Features

In [52]:
X_train = train_rank_df[FEATURE_COLUMNS]
y_train = train_rank_df["label"]

X_valid = valid_rank_df[FEATURE_COLUMNS]
y_valid = valid_rank_df["label"]

## Rebuild Groups

In [53]:
train_groups = (
    train_rank_df
    .groupby("customer_idx")
    .size()
    .values
)

valid_groups = (
    valid_rank_df
    .groupby("customer_idx")
    .size()
    .values
)

In [54]:
print(X_train.shape)
print(X_valid.shape)

print(len(train_groups))
print(len(valid_groups))

print(train_groups[:20])

(193744, 12)
(48370, 12)
41290
10323
[4 4 4 4 4 4 4 4 4 4 4 8 4 4 8 4 4 4 4 4]


## Train LambdaMART

In [55]:
import lightgbm as lgb
import joblib
import json

In [56]:
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",

    n_estimators=300,
    learning_rate=0.05,

    num_leaves=63,
    max_depth=8,

    min_child_samples=20,

    subsample=0.8,
    colsample_bytree=0.8,

    random_state=42,
    n_jobs=-1
)

In [57]:
ranker.fit(
    X_train,
    y_train,

    group=train_groups,

    eval_set=[(X_valid, y_valid)],
    eval_group=[valid_groups],

    eval_at=[5, 10],

    callbacks=[
        lgb.early_stopping(30),
        lgb.log_evaluation(20)
    ]
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.129623 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1553
[LightGBM] [Info] Number of data points in the train set: 193744, number of used features: 12


Training until validation scores don't improve for 30 rounds
[20]	valid_0's ndcg@5: 0.822139	valid_0's ndcg@10: 0.829934
[40]	valid_0's ndcg@5: 0.828204	valid_0's ndcg@10: 0.835666
[60]	valid_0's ndcg@5: 0.833129	valid_0's ndcg@10: 0.840441
[80]	valid_0's ndcg@5: 0.837623	valid_0's ndcg@10: 0.844966
[100]	valid_0's ndcg@5: 0.842432	valid_0's ndcg@10: 0.84953
[120]	valid_0's ndcg@5: 0.846581	valid_0's ndcg@10: 0.853853
[140]	valid_0's ndcg@5: 0.848796	valid_0's ndcg@10: 0.855981
[160]	valid_0's ndcg@5: 0.851376	valid_0's ndcg@10: 0.858699
[180]	valid_0's ndcg@5: 0.852464	valid_0's ndcg@10: 0.859769
[200]	valid_0's ndcg@5: 0.854713	valid_0's ndcg@10: 0.862013
[220]	valid_0's ndcg@5: 0.857458	valid_0's ndcg@10: 0.864832
[240]	valid_0's ndcg@5: 0.860963	valid_0's ndcg@10: 0.868202
[260]	valid_0's ndcg@5: 0.863554	valid_0's ndcg@10: 0.870802
[280]	valid_0's ndcg@5: 0.864864	valid_0's ndcg@10: 0.872165
[300]	valid_0's ndcg@5: 0.86784	valid_0's ndcg@10: 0.874971
Did not meet early stopping. B

,num_leaves,63
,max_depth,8
,learning_rate,0.05
,n_estimators,300
,objective,'lambdarank'
,subsample,0.8
,colsample_bytree,0.8
,random_state,42
,n_jobs,-1
,metric,'ndcg'
,boosting_type,'gbdt'


In [58]:
feature_importance = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance": ranker.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
)

feature_importance

,feature,importance
0,customer_total_spend,4488
2,product_weight_g,2353
9,product_idx,1989
6,popularity_score,1855
4,product_height_cm,1416
3,product_length_cm,1284
5,product_width_cm,1263
8,customer_idx,1209
1,customer_purchase_count,1148
10,category_idx,980


In [59]:
joblib.dump(
    ranker,
    "../artifacts/lambdamart_ranker.pkl"
)

['../artifacts/lambdamart_ranker.pkl']

In [60]:
joblib.dump(
    FEATURE_COLUMNS,
    "../artifacts/ranker_feature_columns.pkl"
)

['../artifacts/ranker_feature_columns.pkl']

In [61]:
joblib.dump(
    customer_encoder,
    "../artifacts/customer_encoder.pkl"
)

joblib.dump(
    product_encoder,
    "../artifacts/product_encoder.pkl"
)

joblib.dump(
    category_encoder,
    "../artifacts/category_encoder.pkl"
)

['../artifacts/category_encoder.pkl']

In [62]:
model_5_metrics = {
    "model_type": "lambdamart",

    "training_rows": int(len(train_rank_df)),
    "validation_rows": int(len(valid_rank_df)),

    "customers_train": int(len(train_groups)),
    "customers_valid": int(len(valid_groups)),

    "features": FEATURE_COLUMNS,

    "best_iteration": int(
        ranker.best_iteration_
    )
}

In [63]:
with open(
    "../artifacts/model_5_metrics.json",
    "w"
) as f:

    json.dump(
        model_5_metrics,
        f,
        indent=4
    )

In [64]:
ranker.best_iteration_

feature_importance.head(10)

,feature,importance
0,customer_total_spend,4488
2,product_weight_g,2353
9,product_idx,1989
6,popularity_score,1855
4,product_height_cm,1416
3,product_length_cm,1284
5,product_width_cm,1263
8,customer_idx,1209
1,customer_purchase_count,1148
10,category_idx,980


In [65]:
feature_importance

,feature,importance
0,customer_total_spend,4488
2,product_weight_g,2353
9,product_idx,1989
6,popularity_score,1855
4,product_height_cm,1416
3,product_length_cm,1284
5,product_width_cm,1263
8,customer_idx,1209
1,customer_purchase_count,1148
10,category_idx,980


## lets again train model by removing customer_idx and product_idx feature

In [66]:
FEATURE_COLUMNS_V2 = [
    "customer_total_spend",
    "customer_purchase_count",

    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",

    "popularity_score",

    "is_favorite_category_match",

    "category_idx",
    "favorite_category_idx"
]

print(len(FEATURE_COLUMNS_V2))

10


In [67]:
X_train_v2 = train_rank_df[FEATURE_COLUMNS_V2]
X_valid_v2 = valid_rank_df[FEATURE_COLUMNS_V2]

y_train_v2 = train_rank_df["label"]
y_valid_v2 = valid_rank_df["label"]

In [68]:
print(X_train_v2.shape)
print(X_valid_v2.shape)

(193744, 10)
(48370, 10)


We are going to test two hypotheses:

Experiment 1:
Remove customer_idx
Remove product_idx
Remove category_idx
Remove favorite_category_idx

Hypothesis:
The model should learn generalizable patterns instead of memorizing IDs.


Experiment 2:
50% hard negatives
50% random negatives

Hypothesis:
is_favorite_category_match should become much more important.

## Create Second Ranker

In [69]:
all_products = train_df["product_id"].unique()

## Mixed Negative Sampling

In [71]:
np.random.seed(42)

negative_samples = []

for row in positive_samples.itertuples(index=False):

    customer_id = row.customer_unique_id
    category = row.product_category_name

    purchased_products = customer_purchases[customer_id]

    # ----------------------------
    # Hard Negatives
    # ----------------------------

    hard_candidates = (
        set(category_products[category])
        - purchased_products
    )

    n_hard = min(2, len(hard_candidates))

    if n_hard > 0:

        hard_negatives = np.random.choice(
            list(hard_candidates),
            size=n_hard,
            replace=False
        )

        for product in hard_negatives:

            negative_samples.append(
                {
                    "customer_unique_id": customer_id,
                    "product_id": product,
                    "product_category_name": category,
                    "label": 0,
                    "negative_type": "hard"
                }
            )

    # ----------------------------
    # Random Negatives
    # ----------------------------

    random_candidates = (
        set(all_products)
        - purchased_products
    )

    n_random = min(2, len(random_candidates))

    if n_random > 0:

        random_negatives = np.random.choice(
            list(random_candidates),
            size=n_random,
            replace=False
        )

        for product in random_negatives:

            negative_samples.append(
                {
                    "customer_unique_id": customer_id,
                    "product_id": product,
                    "product_category_name": None,
                    "label": 0,
                    "negative_type": "random"
                }
            )

In [72]:
negative_samples_df = pd.DataFrame(
    negative_samples
)

negative_samples_df.shape

(242381, 5)

In [73]:
negative_samples_df[
    "negative_type"
].value_counts()

negative_type
random    121296
hard      121085
Name: count, dtype: int64

## Build New Ranking Dataset

In [74]:
ranking_df_v2 = pd.concat(
    [
        positive_samples,
        negative_samples_df.drop(
            columns=["negative_type"]
        )
    ],
    ignore_index=True
)

In [75]:
ranking_df_v2 = ranking_df_v2.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

## Merge Customer Features

In [76]:
ranking_df_v2 = ranking_df_v2.merge(
    customer_features,
    on="customer_unique_id",
    how="left"
)

## Merge Product Features

In [77]:
ranking_df_v2 = ranking_df_v2.merge(
    product_features_rank,
    on="product_id",
    how="left",
    suffixes=("", "_product")
)

## Fix Category Columns

In [78]:
ranking_df_v2["product_category_name"] = (
    ranking_df_v2["product_category_name_product"]
)

ranking_df_v2 = ranking_df_v2.drop(
    columns=["product_category_name_product"]
)

## Interaction Feature

In [79]:
ranking_df_v2[
    "is_favorite_category_match"
] = (
    ranking_df_v2["favorite_category"]
    ==
    ranking_df_v2["product_category_name"]
).astype(int)

## New Feature Set (V2)

In [80]:
FEATURE_COLUMNS_V2 = [
    "customer_total_spend",
    "customer_purchase_count",

    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",

    "popularity_score",

    "is_favorite_category_match"
]

## Train/Validation Split

In [81]:
unique_customers = np.array(
    ranking_df_v2[
        "customer_unique_id"
    ].unique().tolist()
)

In [82]:
train_customers, valid_customers = train_test_split(
    unique_customers,
    test_size=0.2,
    random_state=42
)

In [83]:
train_rank_df_v2 = ranking_df_v2[
    ranking_df_v2["customer_unique_id"]
    .isin(train_customers)
].copy()

valid_rank_df_v2 = ranking_df_v2[
    ranking_df_v2["customer_unique_id"]
    .isin(valid_customers)
].copy()

## Build Groups

In [84]:
from sklearn.preprocessing import LabelEncoder

customer_encoder_v2 = LabelEncoder()

ranking_df_v2["customer_idx"] = (
    customer_encoder_v2.fit_transform(
        ranking_df_v2["customer_unique_id"]
    )
)

In [85]:
unique_customers = np.array(
    ranking_df_v2["customer_unique_id"].unique().tolist()
)

train_customers, valid_customers = train_test_split(
    unique_customers,
    test_size=0.2,
    random_state=42
)

In [86]:
train_rank_df_v2 = ranking_df_v2[
    ranking_df_v2["customer_unique_id"]
    .isin(train_customers)
].copy()

valid_rank_df_v2 = ranking_df_v2[
    ranking_df_v2["customer_unique_id"]
    .isin(valid_customers)
].copy()

In [87]:
train_rank_df_v2 = (
    train_rank_df_v2
    .sort_values("customer_idx")
    .reset_index(drop=True)
)

valid_rank_df_v2 = (
    valid_rank_df_v2
    .sort_values("customer_idx")
    .reset_index(drop=True)
)

In [88]:
train_groups_v2 = (
    train_rank_df_v2
    .groupby("customer_idx")
    .size()
    .values
)

valid_groups_v2 = (
    valid_rank_df_v2
    .groupby("customer_idx")
    .size()
    .values
)

In [89]:
X_train_v2 = train_rank_df_v2[
    FEATURE_COLUMNS_V2
]

y_train_v2 = train_rank_df_v2["label"]

In [90]:
X_valid_v2 = valid_rank_df_v2[
    FEATURE_COLUMNS_V2
]

y_valid_v2 = valid_rank_df_v2["label"]

In [91]:
print(X_train_v2.shape)
print(X_valid_v2.shape)

print(len(train_groups_v2))
print(len(valid_groups_v2))

print(train_groups_v2[:20])

print(
    train_rank_df_v2["label"]
    .value_counts()
)

(242593, 8)
(60436, 8)
41290
10323
[ 5  5  5  5  5  5  5  5  5  5  5 10  5  5 10  5  5  5  5  5]
label
0    194044
1     48549
Name: count, dtype: int64


In [92]:
import lightgbm as lgb
import joblib
import json

In [93]:
ranker_v2 = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",

    n_estimators=2000,
    learning_rate=0.05,

    num_leaves=63,
    max_depth=8,

    min_child_samples=20,

    subsample=0.8,
    colsample_bytree=0.8,

    random_state=42,
    n_jobs=-1
)

In [94]:
ranker_v2.fit(
    X_train_v2,
    y_train_v2,

    group=train_groups_v2,

    eval_set=[(X_valid_v2, y_valid_v2)],
    eval_group=[valid_groups_v2],

    eval_at=[5, 10],

    callbacks=[
        lgb.early_stopping(30),
        lgb.log_evaluation(20)
    ]
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015918 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 910
[LightGBM] [Info] Number of data points in the train set: 242593, number of used features: 8
Training until validation scores don't improve for 30 rounds
[20]	valid_0's ndcg@5: 0.853085	valid_0's ndcg@10: 0.858532
[40]	valid_0's ndcg@5: 0.860961	valid_0's ndcg@10: 0.866391
[60]	valid_0's ndcg@5: 0.863537	valid_0's ndcg@10: 0.868881
[80]	valid_0's ndcg@5: 0.866732	valid_0's ndcg@10: 0.872138
[100]	valid_0's ndcg@5: 0.871225	valid_0's ndcg@10: 0.876439
[120]	valid_0's ndcg@5: 0.874094	valid_0's ndcg@10: 0.879206
[140]	valid_0's ndcg@5: 0.875535	valid_0's ndcg@10: 0.880689
[160]	valid_0's ndcg@5: 0.877346	valid_0's ndcg@10: 0.882615
[180]	valid_0's ndcg@5: 0.880157	valid_0's ndcg@10: 0.885521
[200]	valid_0's ndcg@5: 0.88254	valid_0's ndcg@10: 0.88782
[220]	valid_0's ndcg@5: 0.8854	valid_0's ndcg@1

,num_leaves,63
,max_depth,8
,learning_rate,0.05
,n_estimators,2000
,objective,'lambdarank'
,subsample,0.8
,colsample_bytree,0.8
,random_state,42
,n_jobs,-1
,metric,'ndcg'
,boosting_type,'gbdt'


In [95]:
feature_importance_v2 = pd.DataFrame(
    {
        "feature": FEATURE_COLUMNS_V2,
        "importance": ranker_v2.feature_importances_
    }
)

feature_importance_v2 = (
    feature_importance_v2
    .sort_values(
        "importance",
        ascending=False
    )
)

feature_importance_v2

,feature,importance
0,customer_total_spend,23719
2,product_weight_g,14125
6,popularity_score,10274
4,product_height_cm,9643
3,product_length_cm,9591
5,product_width_cm,8736
1,customer_purchase_count,6010
7,is_favorite_category_match,461


In [96]:
joblib.dump(
    ranker_v2,
    "../artifacts/lambdamart_ranker_v2.pkl"
)

['../artifacts/lambdamart_ranker_v2.pkl']

In [97]:
joblib.dump(
    FEATURE_COLUMNS_V2,
    "../artifacts/ranker_feature_columns_v2.pkl"
)

['../artifacts/ranker_feature_columns_v2.pkl']

In [98]:
model_5_v2_metrics = {
    "model_type": "lambdamart_v2",

    "negative_sampling": {
        "hard_negatives": 2,
        "random_negatives": 2
    },

    "features": FEATURE_COLUMNS_V2,

    "best_iteration": int(
        ranker_v2.best_iteration_
    ),

    "training_rows": int(
        len(train_rank_df_v2)
    ),

    "validation_rows": int(
        len(valid_rank_df_v2)
    )
}

In [99]:
with open(
    "../artifacts/model_5_v2_metrics.json",
    "w"
) as f:

    json.dump(
        model_5_v2_metrics,
        f,
        indent=4
    )

In [100]:
print(ranker_v2.best_iteration_)

print(feature_importance_v2)

1332
                      feature  importance
0        customer_total_spend       23719
2            product_weight_g       14125
6            popularity_score       10274
4           product_height_cm        9643
3           product_length_cm        9591
5            product_width_cm        8736
1     customer_purchase_count        6010
7  is_favorite_category_match         461


In [101]:
joblib.dump(
    ranker_v2,
    "../artifacts/final_ranker.pkl"
)

['../artifacts/final_ranker.pkl']

In [102]:
FEATURE_COLUMNS_FINAL = [
    "customer_total_spend",
    "customer_purchase_count",

    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",

    "popularity_score",

    "is_favorite_category_match"
]

In [103]:
joblib.dump(
    FEATURE_COLUMNS_FINAL,
    "../artifacts/final_ranker_features.pkl"
)

['../artifacts/final_ranker_features.pkl']

In [104]:
final_metrics = {
    "model_name": "LambdaMART_Final",

    "best_iteration": int(ranker_v2.best_iteration_),

    "negative_sampling": {
        "hard": 2,
        "random": 2
    },

    "features": FEATURE_COLUMNS_FINAL,

    "feature_importance": dict(
        zip(
            feature_importance_v2["feature"],
            feature_importance_v2["importance"]
        )
    )
}

In [105]:
with open(
    "../artifacts/final_ranker_metrics.json",
    "w"
) as f:
    json.dump(
        final_metrics,
        f,
        indent=4
    )